In [ ]:

import argparse
import re
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ALGOS = ["gzip", "bzip2", "mfcompress",  "naf", "geco3", "paq8l", "deepdna*"]
colors = ['#1f77b4', '#ff7f0e', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f']

In [2]:
def clean_filename(name: str) -> str:
    """
    Converteix un nom de label en un nom de fitxer segur.
    """
    return (
        name.lower()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(",", "")
        .replace("(", "")
        .replace(")", "")
    )

def clean_title(label: str) -> str:
    return re.sub(r"\s*\([^)]*\)", "", label)

def plot_scatter_by_label(df: pd.DataFrame, output_dir: Path, x_key: str, y_key: str):
    output_dir.mkdir(parents=True, exist_ok=True)

    labels = sorted(df["dataset_label"].dropna().unique())
    title = f"{clean_title(options[x_key])} vs {clean_title(options[y_key])}"

    for label in labels:
        subdf = df[df["dataset_label"] == label].copy()

        if subdf.empty:
            continue

        plt.figure(figsize=(10, 6))

        for algo in sorted(subdf["algorithm"].dropna().unique()):
            algo_df = subdf[subdf["algorithm"] == algo]

            if algo not in ALGOS:
                print(f"[WARNING] Algorithm '{algo}' not in predefined list. Skipping.")
                continue
            
            
            plt.scatter(
                algo_df[x_key],
                algo_df[y_key],
                label=algo,
                alpha=0.8,
                color = colors[ALGOS.index(algo) % len(colors)]
            )
            
        plt.xlabel(options[x_key])
        plt.ylabel(options[y_key])
        plt.title(f"{title} - {label}")

        if y_key == "bits_per_base_seq":
            mean_entropy = subdf["entropy_acgt_bpb"].mean()
            min_entropy = subdf["entropy_acgt_bpb"].min()
            max_entropy = subdf["entropy_acgt_bpb"].max()
            plt.axhline(mean_entropy, color="black", linestyle="-", label=f"H0 mitjana: {mean_entropy:.2f} bpb")
            plt.axhline(min_entropy, color="grey", linestyle=":", label=f"H0 mínima: {min_entropy:.2f} bpb")
            plt.axhline(max_entropy, color="grey", linestyle=":", label=f"H0 màxima: {max_entropy:.2f} bpb")

        # Fer que els eixos comencin sempre a 0
        # plt.xlim(left=0)
        # plt.ylim(bottom=0)

        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.tight_layout()

        out_path = output_dir / f"scatter-{x_key}-{y_key}-{label}.png"
        plt.savefig(out_path, dpi=300)
        plt.close()

        print(f"[OK] Plot saved in: {out_path}")

def plot_scatter_all(df: pd.DataFrame, output_dir: Path, x_key: str, y_key: str):
    output_dir.mkdir(parents=True, exist_ok=True)

    title = f"{clean_title(options[x_key])} vs {clean_title(options[y_key])}"

    plt.figure(figsize=(10, 6))

    for algo in sorted(df["algorithm"].dropna().unique()):
        algo_df = df[df["algorithm"] == algo]
        
        if algo not in ALGOS:
            print(f"[WARNING] Algorithm '{algo}' not in predefined list. Skipping.")
            continue
        plt.scatter(
            algo_df[x_key],
            algo_df[y_key],
            label=algo,
            alpha=0.8,
            color = colors[ALGOS.index(algo) % len(colors)]
        )
        if algo == "genozip":
            print(f"Genozip data points: {algo_df[x_key].values}")
            print(f"Genozip {y_key} values: {algo_df[y_key].values}")
        
    plt.xlabel(options[x_key])
    plt.ylabel(options[y_key])
    plt.title(f"{title} - Tots els datasets")

    if y_key == "bits_per_base_seq":
        mean_entropy = df["entropy_acgt_bpb"].mean()
        min_entropy = df["entropy_acgt_bpb"].min()
        max_entropy = df["entropy_acgt_bpb"].max()
        plt.axhline(mean_entropy, color="black", linestyle="-", label=f"H0 mitjana: {mean_entropy:.2f} bpb")
        plt.axhline(min_entropy, color="grey", linestyle=":", label=f"H0 mínima: {min_entropy:.2f} bpb")
        plt.axhline(max_entropy, color="grey", linestyle=":", label=f"H0 màxima: {max_entropy:.2f} bpb")

    # Fer que els eixos comencin sempre a 0
    # plt.xlim(left=0)
    # plt.ylim(bottom=0)

    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()

    out_path = output_dir / f"scatter-{x_key}-{y_key}-all-nochrX.png"
    plt.savefig(out_path, dpi=300)
    plt.close()

    print(f"[OK] Plot saved in: {out_path}")
    

In [3]:
ALGO_LABELS = {
    "deepdna": "deepdna*"
}

def pretty_algo(name: str) -> str:
    return ALGO_LABELS.get(name, name)

def plot_time_ratio_by_label(df: pd.DataFrame, output_dir: Path):
    """
    Genera un gràfic de barres agrupades per a cada label (dataset)
    mostrant el temps mitjà de compressió i descompressió de cada algorisme.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    labels = df["dataset_label"].dropna().unique()

    for label in labels:
        subdf = df[df["dataset_label"] == label].copy()
        if subdf.empty:
            continue

        # Agrupem per algorisme i calculem la mitjana dels temps
        grouped = subdf.groupby("algorithm")[["compress_seconds", "decompress_seconds"]].mean().reset_index()
        
        # Ens assegurem de mantenir l'ordre d'algorismes de la teva constant si existeixen
        grouped["algo_order"] = grouped["algorithm"].apply(lambda x: ALGOS.index(x) if x in ALGOS else 99)
        grouped = grouped.sort_values("algo_order")

        algos_present = grouped["algorithm"].tolist()
        compress_means = grouped["compress_seconds"].tolist()
        decompress_means = grouped["decompress_seconds"].tolist()

        # Configuració del gràfic de barres agrupades
        x = np.arange(len(algos_present))
        width = 0.35  # Amplada de les barres

        plt.figure(figsize=(12, 6))
        plt.bar(x - width/2, compress_means, width, label='Temps Compressió (s)', color='#2ca02c', alpha=0.8)
        plt.bar(x + width/2, decompress_means, width, label='Temps Descompressió (s)', color='#d62728', alpha=0.8)

        plt.xlabel('Algorisme')
        plt.ylabel('Temps (segons)')
        plt.title(f'Ràtio de Temps de Compressió vs Descompressió - {label}')
        plt.xticks(x, algos_present)
        plt.yscale('log')  # Escala logarítmica molt útil ja que la descompressió sol ser ordres de magnitud més ràpida
        plt.ylabel('Temps en segons (Escala Logarítmica)')
        
        plt.grid(True, which="both", linestyle="--", alpha=0.3)
        plt.legend()
        plt.tight_layout()

        safe_label = clean_filename(label)
        out_path = output_dir / f"time-{safe_label}.png"
        plt.savefig(out_path, dpi=300)
        plt.close()
        print(f"[OK] Time ratio plot saved in: {out_path}")


def plot_status_stacked_bar(df: pd.DataFrame, output_dir: Path):
    """
    Genera un histograma acumulat (gràfic de barres apilades) on es mostra
    per a cada algorisme el recompte de seqüències segons el seu estat (ok, error, etc.).
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    # Creem una taula de contingència (creuada) entre algorisme i estatus
    # Això ens dona exactament quants 'ok', 'error', etc. té cada algoritme
    status_counts = pd.crosstab(df["algorithm"], df["status"])

    # Reordenem les files per seguir l'ordre de la llista ALGOS (només els presents)
    present_algos = [a for a in ALGOS if a in status_counts.index]
    print(present_algos)
    status_counts = status_counts.reindex(present_algos)

    # Dibuixem el gràfic de barres apilades (stacked)
    ax = status_counts.plot(kind='bar', stacked=True, figsize=(10, 6), colormap='viridis', edgecolor='black', alpha=0.85)

    plt.xlabel('Algorisme')
    plt.ylabel('Nombre de seqüències')
    plt.title('Resultats d\'execució per algorisme (Status acumulat)')
    plt.xticks(rotation=45)
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.legend(title='Status', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.tight_layout()

    out_path = output_dir / "status-stacked-histogram.png"
    plt.savefig(out_path, dpi=300)
    plt.close()
    print(f"[OK] Status stacked histogram saved in: {out_path}")

def plot_time_ratio_summary(df: pd.DataFrame, output_dir: Path):
    """
    Genera un ÚNIC gràfic de resum global.
    Calcula el temps mitjà de compressió i descompressió per a cada algorisme
    considerant tots els datasets, afegint barres d'error (desviació estàndard)
    per mostrar la variabilitat entre experiments.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    # 1. Agrupem per algorisme i calculem la mitjana (mean) i la desviació estàndard (std)
    stats = df.groupby("algorithm")[["compress_seconds", "decompress_seconds"]].agg(["mean", "std"]).reset_index()
    
    # Aplanem les columnes multi-índex que genera el .agg()
    stats.columns = ["algorithm", "comp_mean", "comp_std", "decomp_mean", "decomp_std"]

    # 2. Ordenem segons la teva llista d'algorismes de referència
    stats["algo_order"] = stats["algorithm"].apply(lambda x: ALGOS.index(x) if x in ALGOS else 99)
    stats = stats.sort_values("algo_order")

    algos_present = stats["algorithm"].tolist()
    
    # 3. Preparació de les dades
    x = np.arange(len(algos_present))
    width = 0.35  # Amplada de les barres

    plt.figure(figsize=(11, 6))

    # Barres de compressió amb el seu error (capsize dona la línia horitzontal a la barra d'error)
    plt.bar(x - width/2, stats["comp_mean"], width, yerr=stats["comp_std"], 
            label='Temps compressió mitjà (s)', color='#2ca02c', alpha=0.8, capsize=5, edgecolor='black')

    # Barres de descompressió amb el seu error
    plt.bar(x + width/2, stats["decomp_mean"], width, yerr=stats["decomp_std"], 
            label='Temps descompressió mitjà (s)', color='#d62728', alpha=0.8, capsize=5, edgecolor='black')

    # 4. Estètica i configuració
    plt.xlabel('Algorisme')
    plt.ylabel('Temps en segons (escala logarítmica)')
    plt.title('Temps de compressió vs descompressió (mitjana de tots els datasets)')
    
    plt.xticks(x, algos_present, fontsize=10)
    plt.yscale('log')  # L'escala logarítmica segueix sent vital aquí
    
    plt.grid(True, which="both", linestyle="--", alpha=0.3)
    plt.legend(fontsize=10)
    plt.tight_layout()

    # 5. Desat del fitxer únic
    out_path = output_dir / "time-ratio-summary.png"
    plt.savefig(out_path, dpi=300)
    plt.close()
    print(f"[OK] Global time ratio summary saved in: {out_path}")

In [4]:
options = {
    "compress_seconds": "Temps de compressió (s)", 
    "decompress_seconds": "Temps de descompressió (s)", 
    "bits_per_base_seq": "Compressió obtinguda (bpb)",
    "compress_MBps": "Velocitat de compressió (MB/s)",
    "decompress_MBps": "Velocitat de descompressió (MB/s)",
    "compression_percent": "Percentatge de compressió (%)",
    "compression_ratio": "Ratio de compressió",
    "original_size_bytes": "Mida original (bytes)",
}

In [6]:
csv_path = Path.home() / "genomic_benchmark" / "comparative_results.csv"
output_dir = Path.home() / "genomic_benchmark" / "plots" / "escale"
x_var = "compress_seconds"
y_var = "bits_per_base_seq"

df = pd.read_csv(csv_path)

df["algorithm"] = df["algorithm"].apply(pretty_algo)

required_columns = {
    "dataset_id",
    "dataset_label",
    "algorithm",
    "status",
    "entropy_acgt_bpb",
    x_var,
    y_var
}

plot_scatter_by_label(df, output_dir, x_var, y_var)
df_aux = df[df["dataset_id"] != "homo_sapiens_chrX"]
plot_scatter_all(df, output_dir, x_var, y_var)

[WARNING] Algorithm 'geco3' not in predefined list. Skipping.
[WARNING] Algorithm 'genozip' not in predefined list. Skipping.
[WARNING] Algorithm 'paq8l' not in predefined list. Skipping.
[OK] Plot saved in: /home/helen/genomic_benchmark/plots/escale/scatter-compress_seconds-bits_per_base_seq-bacteria.png
[WARNING] Algorithm 'geco3' not in predefined list. Skipping.
[WARNING] Algorithm 'genozip' not in predefined list. Skipping.
[WARNING] Algorithm 'paq8l' not in predefined list. Skipping.
[OK] Plot saved in: /home/helen/genomic_benchmark/plots/escale/scatter-compress_seconds-bits_per_base_seq-chromosome.png
[WARNING] Algorithm 'geco3' not in predefined list. Skipping.
[WARNING] Algorithm 'genozip' not in predefined list. Skipping.
[WARNING] Algorithm 'paq8l' not in predefined list. Skipping.
[OK] Plot saved in: /home/helen/genomic_benchmark/plots/escale/scatter-compress_seconds-bits_per_base_seq-eucariota.png
[WARNING] Algorithm 'deepdna*' not in predefined list. Skipping.
[WARNING] A

In [7]:
#plot_time_ratio_by_label(df, Path.home() / "genomic_benchmark" / "plots" / "all" / "time_histo")
plot_status_stacked_bar(df, Path.home() / "genomic_benchmark" / "plots" / "status")

['gzip', 'bzip2', 'mfcompress', 'naf']
[OK] Status stacked histogram saved in: /home/helen/genomic_benchmark/plots/status/status-stacked-histogram.png


In [100]:
plot_time_ratio_summary(df, Path.home() / "genomic_benchmark" / "plots" / "all" / "time_histo")

[OK] Global time ratio summary saved in: /home/helen/genomic_benchmark/plots/all/time_histo/time-ratio-summary.png


In [76]:
def plot_scatter_broken_axis(df: pd.DataFrame, output_dir: Path, x_key: str, y_key: str):
    output_dir.mkdir(parents=True, exist_ok=True)
    
    # Creem dos subplots costat a costat que comparteixen l'eix Y
    fig, (ax1, ax2) = plt.subplots(1, 2, sharey=True, figsize=(11, 6), gridspec_kw={'width_ratios': [2, 1]})
    fig.suptitle(f"{clean_title(options[x_key])} vs {clean_title(options[y_key])} - Tots els datasets")

    for algo in sorted([x for x in df["algorithm"].dropna().unique() if x != "genozip"]):
        algo_df = df[df["algorithm"] == algo]
        
        # Pintem els mateixos punts a tots dos costats
        for ax in (ax1, ax2):
            ax.scatter(
                algo_df[x_key],
                algo_df[y_key],
                label=algo,
                alpha=0.8,
                color=colors[ALGOS.index(algo) % len(colors)]
            )
            ax.grid(True, alpha=0.3)

    # Definim els límits de cada interval (en bytes)
    ax1.set_xlim(-516240, 7_162_409)
    ax2.set_xlim(27_862_409, 166_270_120)
    # ax3.set_xlim(154_500_000, 166_270_120)

    # Amaguem les línies de divisió interiors
    ax1.spines['right'].set_visible(False)
    ax2.spines['left'].set_visible(False)
    ax1.yaxis.tick_left()
    ax2.yaxis.tick_right() # Els tics del segon gràfic a la dreta (o els amaguem)
    ax2.tick_params(labelright=False) 

    # Dibuixem les típiques línies diagonals de tall (/) a l'eix
    d = .015 
    kwargs = dict(transform=ax1.transAxes, color='black', clip_on=False)
    ax1.plot((1-d, 1+d), (-d, +d), **kwargs)  
    ax1.plot((1-d, 1+d), (1-d, 1+d), **kwargs) 

    kwargs.update(transform=ax2.transAxes)  
    ax2.plot((-d*2, +d*2), (-d, +d), **kwargs)        
    ax2.plot((-d*2, +d*2), (1-d, 1+d), **kwargs)
    
    # Etiquetes globals
    fig.text(0.5, 0.04, options[x_key], ha='center')
    ax1.set_ylabel(options[y_key])
    
    ax2.legend()
    plt.tight_layout()
    
    out_path = output_dir / f"scatter-{x_key}-{y_key}-broken-nogenozip.png"
    plt.savefig(out_path, dpi=300)
    plt.close()

    
    print(f"[OK] Plot broken axis saved in: {out_path}")

In [77]:
plot_scatter_broken_axis(df, output_dir, x_var, y_var)

[OK] Plot broken axis saved in: /home/helen/genomic_benchmark/plots/escale/scatter-original_size_bytes-compression_percent-broken-nogenozip.png
